# Data Mining Assignment: Spark Data Lake Preprocessing

**Student Name:** Akshaya V  
**Roll Number:** LCS2023011  
**Dataset:** sales.csv  
**Environment:** Google Colab with PySpark

## Objective

This notebook creates a simple Bronze and Silver Data Lake structure. The original sales dataset is preserved unchanged in the Bronze layer. Apache Spark is used to inspect and preprocess the data before saving the cleaned result in the Silver layer.

In [1]:
!pip install -q pyspark

## 1. Data Lake Structure

The Bronze layer stores the original dataset without modification. The Silver layer stores the preprocessed data produced using Apache Spark.

In [2]:
import os
import urllib.request

data_lake_dir = "/content/data_lake"
bronze_dir = f"{data_lake_dir}/bronze"
silver_dir = f"{data_lake_dir}/silver"
silver_output_dir = f"{silver_dir}/preprocessed_sales"
bronze_file = f"{bronze_dir}/sales.csv"

os.makedirs(bronze_dir, exist_ok=True)
os.makedirs(silver_dir, exist_ok=True)

dataset_url = (
    "https://raw.githubusercontent.com/"
    "necromancersanju-netizen/data-mining-data-lake-assignment/"
    "main/dataset/sales.csv"
)
urllib.request.urlretrieve(dataset_url, bronze_file)

print("Data Lake structure created successfully.")
print(f"Bronze file: {bronze_file}")
print(f"Silver output directory: {silver_output_dir}")
print(f"Bronze file size: {os.path.getsize(bronze_file)} bytes")

Data Lake structure created successfully.
Bronze file: /content/data_lake/bronze/sales.csv
Silver output directory: /content/data_lake/silver/preprocessed_sales
Bronze file size: 109537 bytes


## 2. Load and Inspect the Bronze Data

A Spark session is created below, and the unchanged Bronze CSV file is loaded into a Spark DataFrame. The schema, sample records, and total record count are displayed.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("SalesDataLakePreprocessing")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark session created successfully.")
print(f"Spark version: {spark.version}")

Spark session created successfully.
Spark version: 4.0.4


In [4]:
bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(bronze_file)
)

raw_count = bronze_df.count()

print("Schema of the Bronze dataset:")
bronze_df.printSchema()
print("Sample Bronze records:")
bronze_df.show(5, truncate=False)
print(f"Total number of Bronze records: {raw_count}")

Schema of the Bronze dataset:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

Sample Bronze records:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+

## 3. Data Quality Inspection

Before preprocessing, Spark is used to identify exact duplicate rows, missing values, text-format inconsistencies, invalid numerical values, and invalid dates.

In [5]:
unique_count = bronze_df.dropDuplicates().count()
duplicate_count = raw_count - unique_count

print(f"Total rows: {raw_count}")
print(f"Unique rows: {unique_count}")
print(f"Exact duplicate rows: {duplicate_count}")
print("Examples of duplicated records:")

(bronze_df
 .groupBy(*bronze_df.columns)
 .count()
 .filter(F.col("count") > 1)
 .show(5, truncate=False))

Total rows: 1000
Unique rows: 990
Exact duplicate rows: 10
Examples of duplicated records:
+--------+-----------+--------------+---------------+---------------+--------+----------+----------------+----------------+----------+-------------+----------+------------+-----+
|order_id|customer_id|customer_name |product        |category       |quantity|unit_price|discount_percent|payment_method  |city      |state        |order_date|order_status|count|
+--------+-----------+--------------+---------------+---------------+--------+----------+----------------+----------------+----------+-------------+----------+------------+-----+
|100692  |C0141      |Varun Agarwal |Electric Kettle|Home Appliances|4       |920       |0               |Net Banking     |Delhi     |Delhi        |2026-08-18|Shipped     |2    |
|100145  |C0007      |Sai Mishra    |Face Wash      |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka    |2025-08-28|Delivered   |2    |
|100297  |C014

In [6]:
missing_value_summary = bronze_df.select([
    F.sum(
        F.when(
            F.col(column).isNull()
            | (F.trim(F.col(column).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(column)
    for column in bronze_df.columns
])

print("Missing-value count for each column:")
missing_value_summary.show(truncate=False)

Missing-value count for each column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [7]:
text_columns = [
    "customer_name", "product", "category",
    "payment_method", "city", "state", "order_status"
]

whitespace_summary = bronze_df.select([
    F.sum(
        F.when(
            F.col(column).isNotNull()
            & (F.col(column) != F.trim(F.col(column))),
            1
        ).otherwise(0)
    ).alias(column)
    for column in text_columns
])

print("Leading or trailing whitespace count:")
whitespace_summary.show(truncate=False)

Leading or trailing whitespace count:
+-------------+-------+--------+--------------+----+-----+------------+
|customer_name|product|category|payment_method|city|state|order_status|
+-------------+-------+--------+--------------+----+-----+------------+
|3            |0      |0       |0             |3   |0    |0           |
+-------------+-------+--------+--------------+----+-----+------------+



In [8]:
print("Capitalization inconsistencies:")

for column in text_columns:
    case_variants = (
        bronze_df
        .filter(F.col(column).isNotNull())
        .select(F.trim(F.col(column)).alias("original_value"))
        .groupBy(F.lower(F.col("original_value")).alias("normalized_value"))
        .agg(
            F.countDistinct("original_value").alias("variant_count"),
            F.collect_set("original_value").alias("variants")
        )
        .filter(F.col("variant_count") > 1)
    )

    if case_variants.count() > 0:
        print(f"\nColumn: {column}")
        case_variants.show(truncate=False)

Capitalization inconsistencies:

Column: category
+----------------+-------------+--------------------------+
|normalized_value|variant_count|variants                  |
+----------------+-------------+--------------------------+
|beauty          |2            |[beauty, Beauty]          |
|electronics     |2            |[ELECTRONICS, Electronics]|
|sports          |3            |[sports, Sports, SPORTS]  |
|clothing        |2            |[CLOTHING, Clothing]      |
+----------------+-------------+--------------------------+


Column: city
+----------------+-------------+------------------------+
|normalized_value|variant_count|variants                |
+----------------+-------------+------------------------+
|delhi           |3            |[DELHI, delhi, Delhi]   |
|hyderabad       |2            |[HYDERABAD, Hyderabad]  |
|jaipur          |2            |[Jaipur, JAIPUR]        |
|coimbatore      |2            |[Coimbatore, coimbatore]|
+----------------+-------------+-----------------

In [9]:
invalid_numeric_summary = bronze_df.select(
    F.sum(F.when(F.col("quantity").isNull() | (F.col("quantity") <= 0), 1)
          .otherwise(0)).alias("invalid_quantity"),
    F.sum(F.when(F.col("unit_price").isNull() | (F.col("unit_price") <= 0), 1)
          .otherwise(0)).alias("invalid_unit_price"),
    F.sum(F.when(
        F.col("discount_percent").isNull()
        | (F.col("discount_percent") < 0)
        | (F.col("discount_percent") > 100), 1
    ).otherwise(0)).alias("invalid_discount_percent")
)

print("Invalid numerical-value counts:")
invalid_numeric_summary.show()

print("Examples of records with invalid numerical values:")
bronze_df.filter(
    (F.col("quantity") <= 0)
    | (F.col("unit_price") <= 0)
    | (F.col("discount_percent") < 0)
    | (F.col("discount_percent") > 100)
).select("order_id", "quantity", "unit_price", "discount_percent")\
 .show(20, truncate=False)

Invalid numerical-value counts:
+----------------+------------------+------------------------+
|invalid_quantity|invalid_unit_price|invalid_discount_percent|
+----------------+------------------+------------------------+
|              12|                 8|                       8|
+----------------+------------------+------------------------+

Examples of records with invalid numerical values:
+--------+--------+----------+----------------+
|order_id|quantity|unit_price|discount_percent|
+--------+--------+----------+----------------+
|100034  |7       |1720      |-5              |
|100102  |7       |0         |40              |
|100114  |0       |2920      |15              |
|100140  |4       |790       |-5              |
|100143  |0       |1030      |15              |
|100170  |6       |0         |0               |
|100198  |-2      |2160      |12              |
|100217  |1       |650       |-5              |
|100235  |7       |-500      |0               |
|100284  |2       |-500  

In [10]:
date_check_df = bronze_df.withColumn(
    "_parsed_order_date",
    F.coalesce(
        F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd")),
        F.try_to_timestamp(F.col("order_date"), F.lit("yyyy/MM/dd"))
    ).cast("date")
)

date_issue_summary = date_check_df.select(
    F.sum(F.when(F.col("_parsed_order_date").isNull(), 1)
          .otherwise(0)).alias("invalid_dates"),
    F.sum(F.when(
        F.col("_parsed_order_date").isNotNull()
        & ~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"), 1
    ).otherwise(0)).alias("inconsistent_but_valid_dates")
)

print("Date-quality summary:")
date_issue_summary.show()
print("Invalid date values:")
date_check_df.filter(F.col("_parsed_order_date").isNull())\
    .select("order_id", "order_date").show(truncate=False)
print("Valid dates using an inconsistent format:")
date_check_df.filter(
    F.col("_parsed_order_date").isNotNull()
    & ~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$")
).select("order_id", "order_date", "_parsed_order_date")\
 .show(truncate=False)

Date-quality summary:
+-------------+----------------------------+
|invalid_dates|inconsistent_but_valid_dates|
+-------------+----------------------------+
|            5|                           1|
+-------------+----------------------------+

Invalid date values:
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100448  |not-a-date|
|100634  |2025-02-30|
|100637  |2026-13-05|
|100969  |09-15-2026|
+--------+----------+

Valid dates using an inconsistent format:
+--------+----------+------------------+
|order_id|order_date|_parsed_order_date|
+--------+----------+------------------+
|100358  |2025/07/14|2025-07-14        |
+--------+----------+------------------+



## 4. Basic Preprocessing — Silver Layer

The preprocessing preserves useful records wherever possible. Exact duplicates are removed; missing descriptive text is labelled as `Unknown`; spaces and obvious capitalization inconsistencies are standardized; invalid quantities and prices are replaced with medians calculated from valid records; invalid discounts are replaced with zero; and invalid dates are stored as null rather than being guessed.

In [11]:
# Remove exact duplicates and handle missing descriptive values
cleaned_df = bronze_df.dropDuplicates().fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown"
})

# Trim all descriptive text fields
for column in text_columns:
    cleaned_df = cleaned_df.withColumn(column, F.trim(F.col(column)))

# Standardize columns that contain obvious capitalization variations
cleaned_df = (
    cleaned_df
    .withColumn("category", F.initcap(F.lower(F.col("category"))))
    .withColumn("city", F.initcap(F.lower(F.col("city"))))
)

print("Duplicates, missing descriptive values, and text formatting handled.")

Duplicates, missing descriptive values, and text formatting handled.


In [12]:
# Calculate medians from valid values instead of hard-coding replacements
quantity_median = int(
    cleaned_df.filter(F.col("quantity") > 0)
    .approxQuantile("quantity", [0.5], 0.0)[0]
)
unit_price_median = int(
    cleaned_df.filter(F.col("unit_price") > 0)
    .approxQuantile("unit_price", [0.5], 0.0)[0]
)

cleaned_df = (
    cleaned_df
    .withColumn(
        "quantity",
        F.when(F.col("quantity") <= 0, quantity_median)
         .otherwise(F.col("quantity"))
    )
    .withColumn(
        "unit_price",
        F.when(F.col("unit_price") <= 0, unit_price_median)
         .otherwise(F.col("unit_price"))
    )
    .withColumn(
        "discount_percent",
        F.when(
            (F.col("discount_percent") < 0)
            | (F.col("discount_percent") > 100), 0
        ).otherwise(F.col("discount_percent"))
    )
)

print(f"Quantity replacement median: {quantity_median}")
print(f"Unit-price replacement median: {unit_price_median}")

Quantity replacement median: 5
Unit-price replacement median: 1850


In [13]:
# Convert accepted dates to one DateType format; invalid dates become null
cleaned_df = cleaned_df.withColumn(
    "order_date",
    F.coalesce(
        F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd")),
        F.try_to_timestamp(F.col("order_date"), F.lit("yyyy/MM/dd"))
    ).cast("date")
)

cleaned_count = cleaned_df.count()
print("Date values standardized to DateType.")
print("Clearly invalid dates were retained as null values.")
print(f"Rows before preprocessing: {raw_count}")
print(f"Rows after preprocessing: {cleaned_count}")

Date values standardized to DateType.
Clearly invalid dates were retained as null values.
Rows before preprocessing: 1000
Rows after preprocessing: 990


In [14]:
# Verify that invalid numerical values were handled
remaining_invalid_numeric = cleaned_df.select(
    F.sum(F.when(F.col("quantity") <= 0, 1).otherwise(0))
     .alias("invalid_quantity"),
    F.sum(F.when(F.col("unit_price") <= 0, 1).otherwise(0))
     .alias("invalid_unit_price"),
    F.sum(F.when(
        (F.col("discount_percent") < 0)
        | (F.col("discount_percent") > 100), 1
    ).otherwise(0)).alias("invalid_discount_percent")
)

print("Remaining invalid numerical values after preprocessing:")
remaining_invalid_numeric.show()
print("Null counts after preprocessing:")
cleaned_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in cleaned_df.columns
]).show(truncate=False)

Remaining invalid numerical values after preprocessing:
+----------------+------------------+------------------------+
|invalid_quantity|invalid_unit_price|invalid_discount_percent|
+----------------+------------------+------------------------+
|               0|                 0|                       0|
+----------------+------------------+------------------------+

Null counts after preprocessing:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |0            |0      |0       |0       |0         |0               |0             |0   |0    |5         |0           |
+--------+-----------+---------

## 5. Save and Verify the Silver Layer

In [15]:
# Save the cleaned data in the Silver layer as CSV
(cleaned_df
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .option("dateFormat", "yyyy-MM-dd")
 .csv(silver_output_dir))

print(f"Silver data saved to: {silver_output_dir}")

Silver data saved to: /content/data_lake/silver/preprocessed_sales


In [16]:
# Read the Silver data back with Spark
silver_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(silver_output_dir)
)

silver_count = silver_df.count()
print(f"Row count before preprocessing: {raw_count}")
print(f"Row count after preprocessing: {cleaned_count}")
print(f"Row count after reading Silver data: {silver_count}")
print("Sample Silver records:")
silver_df.show(10, truncate=False)
print("Silver schema:")
silver_df.printSchema()

Row count before preprocessing: 1000
Row count after preprocessing: 990
Row count after reading Silver data: 990
Sample Silver records:
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Ban

In [17]:
# Create the single CSV file required for GitHub submission
import glob
import shutil

submission_dir = "/content/submissions/LCS2023011_Akshaya_V/silver"
os.makedirs(submission_dir, exist_ok=True)
submission_csv = f"{submission_dir}/sales_silver.csv"

part_files = glob.glob(f"{silver_output_dir}/part-*.csv")
if not part_files:
    raise FileNotFoundError("Spark Silver CSV part file was not found.")

shutil.copyfile(part_files[0], submission_csv)
print(f"Submission CSV created: {submission_csv}")
print(f"Submission CSV size: {os.path.getsize(submission_csv)} bytes")

Submission CSV created: /content/submissions/LCS2023011_Akshaya_V/silver/sales_silver.csv
Submission CSV size: 107538 bytes


## 6. Preprocessing Summary

The following preprocessing operations were performed using Apache Spark:

- Removed exact duplicate rows.
- Replaced missing customer names, payment methods, and cities with `Unknown`.
- Trimmed leading and trailing spaces from descriptive text fields.
- Standardized obvious capitalization inconsistencies in category and city values.
- Replaced non-positive quantities and unit prices with medians calculated from valid records.
- Replaced discount percentages outside the valid 0–100 range with zero.
- Converted valid dates to one consistent date format.
- Represented clearly invalid dates as null rather than inventing values.
- Preserved useful records wherever possible.
- Saved and reloaded the Silver data to verify the output.

## 7. Download the Silver Submission File

Run the following cell after all preceding cells complete successfully.

In [18]:
from google.colab import files
files.download(submission_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>